# Phase 8: Decision Threshold Analysis & Business ROI — Telco Customer Churn

## 1. Executive Summary & Objective
Machine learning classifiers default to a **0.50 decision threshold** ($p \ge 0.50$). However, in customer churn prediction:
- **The cost of a False Negative (missed churner)** is severe (loss of customer lifetime value).
- **The cost of a False Positive (false alarm)** is minimal (the cost of sending a promotional retention offer or proactive customer service call).

### Analytical Objectives:
1. Sweep decision thresholds from $\tau = 0.10$ to $0.90$ on the test set ($N=1,409$).
2. Map Precision, Recall, and F1 trade-offs.
3. Simulate financial return-on-investment (ROI) and net profit under realistic business parameters.
4. Define actionable deployment operating points for different business strategies.

In [1]:
# 2. Environment Setup & Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, precision_recall_curve
)

# Project modules
sys.path.append('..')
from src.feature_engineering import engineer_features
from src.evaluate import analyze_thresholds, simulate_cost_benefit

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_theme(style='whitegrid')

## 3. Data Preparation & Loading Champion Model

In [2]:
# Load processed data
data_path = Path('../data/processed/cleaned_churn_data.csv')
df_clean = pd.read_csv(data_path)
df_feat = engineer_features(df_clean)

X = df_feat.drop(columns=['Churn'])
y = df_feat['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Load champion pipeline
model_path = Path('../models/best_model.pkl')
best_model = joblib.load(model_path)
y_probs = best_model.predict_proba(X_test)[:, 1]

print(f"Loaded champion model from {model_path.resolve()}")
print(f"Test set size: {len(X_test)} samples (Churn count: {y_test.sum()})")

Loaded champion model from D:\ML\customer-churn-prediction\models\best_model.pkl
Test set size: 1409 samples (Churn count: 374)


## 4. Threshold Sweep: Precision, Recall, and F1 Metrics

In [3]:
thresh_df = analyze_thresholds(y_test.values, y_probs)
print("=== CLASSIFICATION PERFORMANCE ACROSS THRESHOLDS ===")
print(thresh_df.to_string(index=False))

# Optimal F1 row
best_f1_row = thresh_df.loc[thresh_df['f1_score'].idxmax()]
print(f"\nOptimal Statistical Balance (Max F1): Threshold = {best_f1_row['threshold']:.2f} (F1 = {best_f1_row['f1_score']:.4f}, Recall = {best_f1_row['recall']:.2%})")

=== CLASSIFICATION PERFORMANCE ACROSS THRESHOLDS ===
 threshold  accuracy  precision  recall  f1_score  customers_flagged
      0.10    0.4961     0.3439  0.9893    0.5103               1076
      0.15    0.5515     0.3694  0.9759    0.5360                988
      0.20    0.5891     0.3882  0.9519    0.5515                917
      0.25    0.6217     0.4079  0.9412    0.5691                863
      0.30    0.6579     0.4323  0.9225    0.5887                798
      0.35    0.6835     0.4514  0.8930    0.5996                740
      0.40    0.7048     0.4690  0.8503    0.6046                678
      0.45    0.7296     0.4945  0.8342    0.6209                631
      0.50    0.7459     0.5136  0.8075    0.6279                588
      0.55    0.7587     0.5312  0.7727    0.6296                544
      0.60    0.7779     0.5648  0.7112    0.6296                471
      0.65    0.7956     0.6091  0.6417    0.6250                394
      0.70    0.7999     0.6411  0.5588    0.5971 

## 5. Financial Cost-Benefit & ROI Simulation

### Simulation Parameters:
- **Value of Retaining a Customer ($V_{\text{save}}$)**: $\$300$ (representing ~5 months of average recurring billing).
- **Cost of Retention Intervention ($C_{\text{outreach}}$)**: $\$30$ per customer contacted.
- **Retention Success Rate ($P_{\text{success}}$)**: $40\%$ of contacted churners accept the offer and remain.

In [4]:
cost_df = simulate_cost_benefit(
    y_test.values, y_probs, clv_saved=300.0, cost_intervention=30.0, success_rate=0.40
)
print("=== BUSINESS COST-BENEFIT & ROI SIMULATION ===")
print(cost_df.to_string(index=False))

# Maximum profit threshold
max_profit_row = cost_df.loc[cost_df['net_profit'].idxmax()]
print(f"\nMaximum Net Profit: Threshold = {max_profit_row['threshold']:.2f} -> Net Profit = ${max_profit_row['net_profit']:,.2f} (ROI: {max_profit_row['roi_pct']:.1f}%)")

=== BUSINESS COST-BENEFIT & ROI SIMULATION ===
 threshold  targeted  true_churners_caught  churners_saved  campaign_cost  revenue_saved  net_profit  roi_pct
      0.10      1076                   370           148.0        32280.0        44400.0     12120.0     37.5
      0.15       988                   365           146.0        29640.0        43800.0     14160.0     47.8
      0.20       917                   356           142.4        27510.0        42720.0     15210.0     55.3
      0.25       863                   352           140.8        25890.0        42240.0     16350.0     63.2
      0.30       798                   345           138.0        23940.0        41400.0     17460.0     72.9
      0.35       740                   334           133.6        22200.0        40080.0     17880.0     80.5
      0.40       678                   318           127.2        20340.0        38160.0     17820.0     87.6
      0.45       631                   312           124.8        18930.0

## 6. Strategic Business Operating Points
Mapping out three actionable deployment policies for the retention team.

In [5]:
scenarios = [
    {
        'Strategy': 'Aggressive Retention (Catch Maximum Churners)',
        'Threshold': 0.35,
        'Recall': f"{thresh_df.loc[thresh_df['threshold']==0.35, 'recall'].values[0]:.1%}",
        'Precision': f"{thresh_df.loc[thresh_df['threshold']==0.35, 'precision'].values[0]:.1%}",
        'Targeted': int(cost_df.loc[cost_df['threshold']==0.35, 'targeted'].values[0]),
        'Campaign Cost': f"${cost_df.loc[cost_df['threshold']==0.35, 'campaign_cost'].values[0]:,.0f}",
        'Net Profit': f"${cost_df.loc[cost_df['threshold']==0.35, 'net_profit'].values[0]:,.0f}",
        'ROI': f"{cost_df.loc[cost_df['threshold']==0.35, 'roi_pct'].values[0]:.1f}%"
    },
    {
        'Strategy': 'Profit Maximization (Optimal Economic Return)',
        'Threshold': 0.50,
        'Recall': f"{thresh_df.loc[thresh_df['threshold']==0.50, 'recall'].values[0]:.1%}",
        'Precision': f"{thresh_df.loc[thresh_df['threshold']==0.50, 'precision'].values[0]:.1%}",
        'Targeted': int(cost_df.loc[cost_df['threshold']==0.50, 'targeted'].values[0]),
        'Campaign Cost': f"${cost_df.loc[cost_df['threshold']==0.50, 'campaign_cost'].values[0]:,.0f}",
        'Net Profit': f"${cost_df.loc[cost_df['threshold']==0.50, 'net_profit'].values[0]:,.0f}",
        'ROI': f"{cost_df.loc[cost_df['threshold']==0.50, 'roi_pct'].values[0]:.1f}%"
    },
    {
        'Strategy': 'Budget-Constrained (High Precision Outreach)',
        'Threshold': 0.65,
        'Recall': f"{thresh_df.loc[thresh_df['threshold']==0.65, 'recall'].values[0]:.1%}",
        'Precision': f"{thresh_df.loc[thresh_df['threshold']==0.65, 'precision'].values[0]:.1%}",
        'Targeted': int(cost_df.loc[cost_df['threshold']==0.65, 'targeted'].values[0]),
        'Campaign Cost': f"${cost_df.loc[cost_df['threshold']==0.65, 'campaign_cost'].values[0]:,.0f}",
        'Net Profit': f"${cost_df.loc[cost_df['threshold']==0.65, 'net_profit'].values[0]:,.0f}",
        'ROI': f"{cost_df.loc[cost_df['threshold']==0.65, 'roi_pct'].values[0]:.1f}%"
    }
]
strategy_df = pd.DataFrame(scenarios)
print("=== STRATEGIC BUSINESS PERSONAS ===")
print(strategy_df.to_string(index=False))

=== STRATEGIC BUSINESS PERSONAS ===
                                     Strategy  Threshold Recall Precision  Targeted Campaign Cost Net Profit    ROI
Aggressive Retention (Catch Maximum Churners)       0.35  89.3%     45.1%       740       $22,200    $17,880  80.5%
Profit Maximization (Optimal Economic Return)       0.50  80.8%     51.4%       588       $17,640    $18,600 105.4%
 Budget-Constrained (High Precision Outreach)       0.65  64.2%     60.9%       394       $11,820    $16,980 143.7%


## 7. Conclusions & Handoff to Phase 9

1. **Operational Tunability**: The model is not locked into an arbitrary 50% cutoff; stakeholders can dynamically select the threshold based on campaign budget and urgency.
2. **Primary Operating Threshold**: The **0.50** cutoff delivers the **maximum dollar profit ($18,600)** while maintaining an exceptional **80.75% Recall**.
3. **Handoff to Phase 9 (Explainability)**: With the champion model verified and threshold operating points established, Phase 9 will use **SHAP (SHapley Additive exPlanations)** to uncover global feature importance and provide local customer-level explanations.